# Notebook 2: Comparing Two Sequences with Pairwise Alignment

**Chem 274A · Lab Week 11 · Sequence Similarity Networks** · about 20 minutes

<div class="alert alert-block alert-info">
<h3>Learning goals</h3>

- Understand what a sequence alignment is and how it is scored.
- Use Biopython's `PairwiseAligner` class with a real substitution matrix (BLOSUM62).
- See why raw alignment scores need to be **normalized** before comparing them.
</div>

In [ ]:
from Bio import SeqIO, Align
from Bio.Align import substitution_matrices

records = list(SeqIO.parse("data/globins.fasta", "fasta"))
seq_dict = SeqIO.to_dict(records)

## What is an alignment?

To compare two sequences, we line them up so that matching letters sit in the same column.
Sometimes we need to slide part of one sequence over by inserting a **gap** (`-`):

```
G A T T A C A          G A T T A C A
| | | . . . .    vs.   | | | - | | |
G A T A C A -          G A T - A C A     (made-up example)
```

Each alignment gets a **score**: points for matches, penalties for mismatches and gaps.
The aligner's job is to find the alignment with the **best possible score**. It uses dynamic programming
(the Needleman–Wunsch and Smith–Waterman algorithms). You won't need to code that today, because Biopython does it for you.

## `PairwiseAligner`

`PairwiseAligner` is a class. You create an aligner object, set its attributes (the scoring rules),
then call its methods.

In [ ]:
aligner = Align.PairwiseAligner()
print(aligner)   # shows every scoring setting

In [ ]:
print(aligner.score("GATTACA", "GCATGCA"))       # just the best score (fast)

alignments = aligner.align("GATTACA", "GCATGCA")  # the actual alignment(s)
print(alignments[0])

How to read the printed alignment:

- `|` means the two letters are identical,
- `.` means a mismatch,
- `-` means a gap.

## Substitution matrices: some mismatches are better than others

For proteins, "match = +1, mismatch = 0" is too simple. Swapping one amino acid for a chemically similar one
usually barely changes a protein, but swapping in one with very different chemistry can wreck it.
So when we compare two related proteins, some mismatches are much more common than others.

Each letter in a protein sequence stands for one of the 20 amino acids. They all share the same backbone
(an amine and a carboxylic acid) and differ only in their **side chain**.
Keep this chart open in another tab: **[A Guide to the Twenty Common Amino Acids](https://i0.wp.com/www.compoundchem.com/wp-content/uploads/2014/09/20-Common-Amino-Acids-v3.png?ssl=1)**. It shows each amino acid's structure, one-letter code, and side-chain category.

A **substitution matrix** gives a score for every pair of amino acids, based on how often that swap is seen
in real families of related proteins:

- **positive**: this swap happens *more* often than chance, so it's a "safe" substitution,
- **zero**: about as often as chance,
- **negative**: *less* often than chance, so nature avoids it.

Matching an amino acid with itself is always positive. **BLOSUM62** is the default in BLAST and the most common
choice. Its scores run from **−4 to +11**.

<div class="alert alert-block alert-warning">
<h3>Predict, then check</h3>

Before you run the next cell, **guess** whether each of these BLOSUM62 scores is positive, zero or negative:

**L/I · F/Y · D/E · K/R · K/D · L/D · A/G · L/L · W/W**

Use the amino acid chart. Similar side chains (both hydrophobic, both acidic, both aromatic...) usually score a little
above zero, and very different side chains score negative. An amino acid matched with itself is always positive.
Compare guesses with your neighbors, then run the cell, which looks up each pair as `blosum62[a][b]`.
</div>

In [ ]:
blosum62 = substitution_matrices.load("BLOSUM62")

pairs = ["LI", "FY", "DE", "KR", "KD", "LD", "AG", "LL", "WW"]
for a, b in pairs:   # each two-letter string unpacks into its two letters
    print(f"{a}/{b}: {blosum62[a][b]:.0f}")

<div class="alert alert-block alert-warning">
<h3>Think about it</h3>

1. What do the two amino acids in each **positive** pair have in common? (Use the color categories on the chart:
   aliphatic, aromatic, acidic, basic...)
2. Lysine/aspartate (K/D) have opposite charges, yet they score −1, while leucine/aspartate (L/D) score −4. Why might
   swapping two charged amino acids be less harmful? *Hint:* where in a folded protein do you find charged vs.
   hydrophobic side chains?
3. W/W scores 11 but L/L only 4. *Hint:* tryptophan is one of the rarest amino acids, and leucine is the most common.
</div>

## Aligning real proteins

Now let's set up an aligner the way protein scientists would:

- `mode="local"` finds the best-matching *region* instead of forcing the whole sequences to align end to end (this is what BLAST does),
- `substitution_matrix=blosum62`,
- gap penalties close to BLAST's defaults.

In [ ]:
aligner = Align.PairwiseAligner(
    mode="local",
    substitution_matrix=blosum62,
    open_gap_score=-11,
    extend_gap_score=-1,
)

hba_human = seq_dict["sp|P69905|HBA_HUMAN"].seq
hbb_human = seq_dict["sp|P68871|HBB_HUMAN"].seq

alignment = aligner.align(hba_human, hbb_human)[0]
print(alignment)
print("score:", alignment.score)

### Who is most similar to human hemoglobin α?

The cell below scores human hemoglobin α (`hba_human`) against five other globins and prints them from most to least
similar:

| ID | Protein |
|---|---|
| sp\|P69907\|HBA_PANTR | chimpanzee hemoglobin α |
| sp\|P68871\|HBB_HUMAN | human hemoglobin β |
| sp\|P02144\|MYG_HUMAN | human myoglobin |
| sp\|Q9NPG2\|NGB_HUMAN | human neuroglobin |
| sp\|P02238\|LGB3_SOYBN | soybean leghemoglobin |

Does the order make sense biologically? Think about which proteins come from the same species, which do the same job,
and how far apart humans and plants are on the tree of life.

In [ ]:
others = ["sp|P69907|HBA_PANTR", "sp|P68871|HBB_HUMAN", "sp|P02144|MYG_HUMAN",
          "sp|Q9NPG2|NGB_HUMAN", "sp|P02238|LGB3_SOYBN"]

scores = {}
for other in others:
    scores[other] = aligner.score(hba_human, seq_dict[other].seq)

for other in sorted(scores, key=scores.get, reverse=True):
    print(other, "score:", scores[other])

## Percent identity

Scores are hard to interpret on their own. A number people *can* read is **percent identity**:
the fraction of aligned columns where the two amino acids are identical. `alignment.counts()` counts them:

In [ ]:
alignment = aligner.align(hba_human, hbb_human)[0]   # human α vs. human β
counts = alignment.counts()
print("identities:", counts.identities)
print("mismatches:", counts.mismatches)
print("gaps:      ", counts.gaps)
print("columns:   ", alignment.length)

This function packages that up. `100 * identities / alignment.length` is the same definition BLAST uses.


In [ ]:
def percent_identity(seq_a, seq_b):
    alignment = aligner.align(seq_a, seq_b)[0]
    return 100 * alignment.counts().identities / alignment.length

hba_chimp = seq_dict["sp|P69907|HBA_PANTR"].seq   # chimpanzee hemoglobin α
print(percent_identity(hba_human, hba_chimp))   # human and chimp α are identical
print(percent_identity(hba_human, hbb_human))   # α and β, about 43% identical

## The problem with raw scores

Every matching column *adds* to the score, so **longer sequences get higher scores just for being long**.
Remember from Notebook 1 that cytoglobins are the longest globins. Look at what happens when we align each
sequence with itself (a perfect match):

In [ ]:
for name in ["sp|P69905|HBA_HUMAN", "sp|P02144|MYG_HUMAN", "sp|Q8WWM9|CYGB_HUMAN"]:
    seq = seq_dict[name].seq
    print(f"{name:22s} length {len(seq):3d}   self-score {aligner.score(seq, seq):6.1f}")

A perfect match scores about 730 for hemoglobin α but about 980 for cytoglobin. So if we picked one raw-score cutoff for the
whole network, long proteins (like cytoglobin) would connect more easily than short ones, even when they are no more alike.

The fix is to **normalize**: divide the score by the best score that pair could possibly get.
A simple version is:

$$
\text{normalized score}(a, b) = \frac{\text{score}(a, b)}{\min\big(\text{score}(a, a),\ \text{score}(b, b)\big)}
$$

This is **1.0** for identical sequences and close to **0** for unrelated ones, no matter how long they are.

> Real tools solve the same problem in a more statistical way. BLAST reports **bit scores** and **E-values**
> (the number of hits this good you would expect by pure chance). EFI-EST builds its networks from those.
> You'll meet E-values in the bonus notebook.

<div class="alert alert-block alert-success">
<h3>Exercise: Write <code>normalized_score</code></h3>

Write `normalized_score(seq_a, seq_b)` using the formula above, then run the checks at the bottom of the cell:
- human α vs itself should be `1.0`,
- human α vs chimp α should be `1.0` (they're identical),
- human α vs soybean leghemoglobin should be small.
</div>

In [ ]:
def normalized_score(seq_a, seq_b):
    # Your code here
    pass

lgb_soybean = seq_dict["sp|P02238|LGB3_SOYBN"].seq   # soybean leghemoglobin
print(normalized_score(hba_human, hba_human))    # should be 1.0
print(normalized_score(hba_human, hba_chimp))    # should be 1.0
print(normalized_score(hba_human, lgb_soybean))  # should be small

## Summary

- `PairwiseAligner` is a configurable class: set the mode, substitution matrix and gap scores, then call `.score()` or `.align()`.
- BLOSUM62 rewards chemically similar swaps.
- Raw scores grow with length, so we **normalize** them before choosing a cutoff.

**Next:** Notebook 3, where we score every pair and build the network.